# ComfyUI on Colab — Personalised Launcher

## 1. Mount Google Drive (optional, for prefs only)

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PREFS_PATH = "/content/drive/MyDrive/.comfyui-colab/prefs.json"
    print("Drive mounted. Prefs path:", PREFS_PATH)
except Exception as e:
    PREFS_PATH = "/content/.comfyui-colab-prefs.json"
    print(f"Drive not available ({type(e).__name__}); prefs will live at {PREFS_PATH}")


## 2. Detect GPU and check torch compatibility

In [ ]:
import subprocess, sys

# Use stdlib only — helpers not yet on sys.path at this point in the notebook.
def _cc():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True
        ).strip().splitlines()[0]
        name, cc = [s.strip() for s in out.split(",", 1)]
    except Exception:
        return "CPU", 0
    try:
        maj, mn = cc.split(".")
        return name, int(maj) * 10 + int(mn)
    except Exception:
        return name, 0

GPU_NAME, GPU_CC = _cc()
print(f"GPU: {GPU_NAME}  sm_{GPU_CC if GPU_CC else 'none'}")

import torch
ARCH_LIST = list(torch.cuda.get_arch_list()) if torch.cuda.is_available() else []
print(f"torch {torch.__version__} archs: {ARCH_LIST}")

NEEDS_REINSTALL = GPU_CC != 0 and f"sm_{GPU_CC}" not in ARCH_LIST
if NEEDS_REINSTALL:
    print(f"WARNING: sm_{GPU_CC} not in preinstalled torch; reinstalling cu128…")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "torch==2.11.0", "torchvision==0.26.0", "torchaudio==2.11.0",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ])
    print("
REINSTALLED. You MUST choose Runtime → Restart session before running the next cell.")
else:
    print("Preinstalled torch is fine; no reinstall needed.")


## 3. Load secrets (HF_TOKEN, CIVITAI_TOKEN)

In [ ]:
import os, getpass, sys

def _load(name: str):
    mod = sys.modules.get("google.colab")
    if mod is None:
        try:
            import google.colab as mod  # type: ignore
        except ImportError:
            mod = None
    if mod is not None and hasattr(mod, "userdata"):
        try:
            val = mod.userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    try:
        val = getpass.getpass(f"{name} (hidden, or press Enter to skip): ")
    except Exception:
        return None
    return val or None

HF_TOKEN = _load("HF_TOKEN")
CIVITAI_TOKEN = _load("CIVITAI_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if CIVITAI_TOKEN:
    os.environ["CIVITAI_TOKEN"] = CIVITAI_TOKEN
print(f"HF_TOKEN: {'set' if HF_TOKEN else 'missing'}   CIVITAI_TOKEN: {'set' if CIVITAI_TOKEN else 'missing'}")
del HF_TOKEN, CIVITAI_TOKEN  # drop from IPython _oh history


## 4. Install lightweight dependencies

In [ ]:
!apt-get install -y -qq aria2 >/dev/null
!pip install -q --upgrade ipywidgets huggingface_hub
!pip install -q --upgrade "huggingface_hub[cli]"  # provides the `hf` CLI
print("Deps installed.")


## 5. Clone the ComfyUI repo

In [ ]:
import os, subprocess
REPO_URL = os.environ.get("COMFYUI_COLAB_REPO_URL", "https://github.com/samsam27/ComfyUI.git")
REPO_DIR = "/content/ComfyUI"
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo ready at", REPO_DIR)


## 6. Pipeline and LoRA selector

In [ ]:
import json, os
from pathlib import Path
import ipywidgets as W
from IPython.display import display

from colab.launcher_helpers import load_registry, list_pipelines, pipeline_loras, load_prefs

REGISTRY_PATH = "/content/ComfyUI/scripts/model_registry.json"
registry = load_registry(REGISTRY_PATH)
pipelines = list_pipelines(registry)
prefs = load_prefs(PREFS_PATH)
selected_prev = set(prefs.get("pipelines", []))
lora_prev = {k: set(v) for k, v in prefs.get("loras", {}).items()}

STATE = {"pipelines": set(), "loras": {}, "prefs_path": PREFS_PATH}

pipeline_boxes = []
lora_widgets = {}

for p in pipelines:
    cb = W.Checkbox(
        value=(p["key"] in selected_prev),
        description=f"{p['display_name']}  ({p['total_size_gb']} GB, {p['model_count']} models, {p['lora_count']} LoRAs)",
        indent=False,
        layout=W.Layout(width="100%"),
    )
    pipeline_boxes.append((p["key"], cb))

def _update_lora_pane(*_):
    STATE["pipelines"] = {k for k, cb in pipeline_boxes if cb.value}
    children = []
    STATE["loras"] = {}
    for key in sorted(STATE["pipelines"]):
        loras = pipeline_loras(registry, key)
        if not loras:
            continue
        options = [(f"{l['display_name']} ({l.get('size_gb', 0):.2f} GB)", l["filename"]) for l in loras]
        preselect = [fn for _disp, fn in options if fn in lora_prev.get(key, set())]
        sm = W.SelectMultiple(
            options=options, value=tuple(preselect),
            description=key, rows=min(6, len(options)),
            layout=W.Layout(width="100%"),
        )
        lora_widgets[key] = sm
        STATE["loras"][key] = set(preselect)
        def _on_change(change, _key=key):
            STATE["loras"][_key] = set(change["new"])
        sm.observe(_on_change, names="value")
        children.append(sm)
    lora_pane.children = children

for _k, cb in pipeline_boxes:
    cb.observe(_update_lora_pane, names="value")

lora_pane = W.VBox([], layout=W.Layout(width="50%"))
left = W.VBox([W.HTML("<b>Pipelines</b>")] + [cb for _k, cb in pipeline_boxes],
              layout=W.Layout(width="50%"))
right = W.VBox([W.HTML("<b>LoRAs (per selected pipeline)</b>"), lora_pane])

launch_btn = W.Button(description="Launch ComfyUI", button_style="success", icon="rocket")
status = W.HTML("<i>Pick at least one pipeline, then click Launch.</i>")

def _on_launch(_btn):
    if not STATE["pipelines"]:
        status.value = "<span style='color:#a00'>Select at least one pipeline first.</span>"
        return
    status.value = "<b>Launching…</b> scroll to section 7 for progress."
    from colab.launcher_helpers import save_prefs
    save_prefs(PREFS_PATH, {
        "pipelines": sorted(STATE["pipelines"]),
        "loras": {k: sorted(v) for k, v in STATE["loras"].items()},
    })
    launch_btn.disabled = True
    STATE["launch_triggered"] = True

launch_btn.on_click(_on_launch)
_update_lora_pane()  # initial render
display(W.VBox([W.HBox([left, right]), launch_btn, status]))


## 7. Launch ComfyUI (install reqs, download weights, start main.py)

In [ ]:
# filled in a later task

## 8. Public URL via Cloudflared quick tunnel

In [ ]:
# filled in a later task